# Hyperparameter Tuning

Using Keras Tuner to find the best configuration.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import keras_tuner as kt
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pd.read_csv("final-data.csv")
df["Week Endings"] = pd.to_datetime(df["Week Endings"])
df = df.sort_values("Week Endings").reset_index(drop=True)

def create_dataset(dataset, look_back=12):
    X, Y = [], []
    for i in range(len(dataset)-look_back):
        X.append(dataset[i:(i+look_back), 0])
        Y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(Y)

price_data = df["Avg Ticket Price ($)"].values.astype("float32").reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
price_scaled = scaler.fit_transform(price_data)

cutoff = pd.to_datetime("2017-04-01")
train = price_scaled[df["Week Endings"] < cutoff]
X_train, y_train = create_dataset(train, look_back=12)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

In [ ]:
def build_model(hp):
    model = keras.Sequential()
    model.add(keras.layers.SimpleRNN(
        units=hp.Int("units", min_value=16, max_value=64, step=16),
        input_shape=(12, 1)
    ))
    model.add(keras.layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice("learning_rate", values=[1e-2, 1e-3, 1e-4])
        ),
        loss="mse"
    )
    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_loss",
    max_trials=5,
    executions_per_trial=1,
    directory="tuner_dir",
    project_name="broadway_lion_king"
)

tuner.search(X_train, y_train, epochs=10, validation_split=0.2)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best units: {best_hps.get('units')}")
print(f"Best learning rate: {best_hps.get('learning_rate')}")